# Chest X-Ray Pneumonia Detector

This notebook documents a reproducible binary image-classification workflow. It requires a locally available dataset arranged as `train/val/test` with `NORMAL` and `PNEUMONIA` class folders. The dataset used for the documented smoke run was downloaded with `kagglehub.dataset_download("ghost5612/chest-x-ray-images-normal-and-pneumonia")`. No full patient-image dataset is committed to this repository.

**Medical-use boundary:** this is an educational model-development workflow, not a clinical diagnostic system.

In [ ]:
from pathlib import Path\nimport copy\n# Optional download step used for the documented run:\n# import kagglehub\n# DATA_ROOT = Path(kagglehub.dataset_download(\"ghost5612/chest-x-ray-images-normal-and-pneumonia\")) / \"chest_xray\"\nimport torch
import torch.nn as nn
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

from src.data import build_dataloaders
from src.model import build_model

DATA_ROOT = Path('../data/chest_xray')
ARTIFACTS = Path('../artifacts')
ARTIFACTS.mkdir(exist_ok=True)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
BATCH_SIZE = 32
EPOCHS = 5
print({'device': str(DEVICE), 'data_root': str(DATA_ROOT.resolve())})

In [ ]:
if not DATA_ROOT.exists():
    raise FileNotFoundError(
        f'{DATA_ROOT.resolve()} is missing. Download an appropriately licensed dataset, '
        'place it under data/chest_xray, and rerun this cell.'
    )
loaders = build_dataloaders(DATA_ROOT, batch_size=BATCH_SIZE)
print({split: len(loader.dataset) for split, loader in loaders.items()})
print('class_to_idx:', loaders['train'].dataset.class_to_idx)

## Model and training objective

The model uses a ResNet-18 convolutional backbone with a two-class output head. The default construction does not download weights; set `pretrained=True` only after reviewing the weight license and experiment plan. For a serious experiment, address class imbalance explicitly and keep the test set untouched until final evaluation.

In [ ]:
model = build_model(num_classes=2, pretrained=False).to(DEVICE)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
print(model.fc)

In [ ]:
def run_epoch(loader, training=False):
    model.train(training)
    total_loss, total = 0.0, 0
    for images, labels in loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        if training:
            optimizer.zero_grad(set_to_none=True)
        logits = model(images)
        loss = criterion(logits, labels)
        if training:
            loss.backward()
            optimizer.step()
        total_loss += loss.item() * labels.size(0)
        total += labels.size(0)
    return total_loss / max(total, 1)

best_state, best_val = None, float('inf')
for epoch in range(EPOCHS):
    train_loss = run_epoch(loaders['train'], training=True)
    val_loss = run_epoch(loaders['val'])
    if val_loss < best_val:
        best_val, best_state = val_loss, copy.deepcopy(model.state_dict())
    print(f'epoch={epoch+1} train_loss={train_loss:.4f} val_loss={val_loss:.4f}')

if best_state is not None:
    model.load_state_dict(best_state)
    torch.save(model.state_dict(), ARTIFACTS / 'pneumonia_detector.pt')
    print('saved:', ARTIFACTS / 'pneumonia_detector.pt')

In [ ]:
model.eval()
y_true, y_pred, y_score = [], [], []
with torch.no_grad():
    for images, labels in loaders['test']:
        probs = torch.softmax(model(images.to(DEVICE)), dim=1)[:, 1].cpu()
        y_true.extend(labels.tolist())
        y_score.extend(probs.tolist())
        y_pred.extend((probs >= 0.5).int().tolist())

print(classification_report(y_true, y_pred, target_names=['NORMAL', 'PNEUMONIA'], zero_division=0))
print('confusion_matrix:\n', confusion_matrix(y_true, y_pred))
if len(set(y_true)) == 2:
    print('roc_auc:', roc_auc_score(y_true, y_score))
else:
    print('ROC-AUC skipped because the test split has one observed class.')

## Interpretation checklist

Do not treat the metrics above as portable clinical evidence. Before presenting a benchmark, document the dataset source and license, patient-level split policy, class counts, preprocessing, threshold selection, sensitivity, specificity, calibration, error examples, and known demographic or site-level limitations. Add a model card and a versioned evaluation report for any deployed checkpoint.